# Step 4 — Batch Auto Draft (NO MANUAL APPROVAL)

**Purpose:** Automatically generate an **unapproved text-only preview** of each eligible Table from the six v4 Adaptation Plans. You do **not** inspect or approve individual operations.

This notebook uses a *different* pathway from the approved-only executor (Notebook 07). It **never** populates a human approval packet or marks any operation approved. Only exact bookmark-anchored titles and SAP-quote-verified treatment-arm labels are eligible. All remaining layout, row/column, footnote, denominator, statistical method and clinical checks are exported as an **exception report**.

**Input:** `step4_adaptation_plan_outputs.zip` from Notebook 06. No DeepSeek calls, no review-packet editing and no source Library modifications. **Output:** a ZIP with diagnostic JSONs and, where safe, a copy-only Word preview.

For the current six-table pilot, it is normal to see AE as the only table with eligible edits. The other five are blocked or require a layout rebuild. This does **not** create six finished Shells.

## 1 · Clone the official development branch and install TFL Agent

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
GIT_REF = "feature/official-library-colab"
PROJECT_ROOT = Path("/content/tfl-official-step4-replay")
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_REF])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                           REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git checkout")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tfl_agent import TFLSession
from tfl_agent.tools.library_loader import load_library
from tfl_agent.agent.step3_retriever import retrieve_templates, template_selection_decision
from tfl_agent.tools.io_utils import write_json
from IPython.display import display, FileLink
import pandas as pd
import json, zipfile, shutil, traceback

git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Using Agent + official Library branch:", GIT_REF, "at", git_sha)
print("Python package:", __import__("tfl_agent").__file__)

## 2 · Upload the Adaptation Plan ZIP generated by Colab 06

In [ ]:
from google.colab import files
from tfl_agent.agent.auto_text_draft import build_unapproved_auto_text_draft
from pathlib import Path

PLAN_ZIP_NAME = "step4_adaptation_plan_outputs.zip"
search_paths = [Path.cwd() / PLAN_ZIP_NAME, Path("/content") / PLAN_ZIP_NAME]
PLAN_ZIP = next((p for p in search_paths if p.is_file()), None)
if PLAN_ZIP is None:
    print("Upload the Adaptation Plan ZIP from Notebook 06.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No ZIP selected")
    name = PLAN_ZIP_NAME if PLAN_ZIP_NAME in uploaded else next(iter(uploaded))
    PLAN_ZIP = next((p for p in [Path.cwd()/name, Path("/content")/name] if p.is_file()), None)
assert PLAN_ZIP is not None and PLAN_ZIP.suffix.lower() == ".zip"

INPUT_ROOT = Path("/content/step4_auto_preview_input")
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PLAN_ZIP) as archive:
    for member in archive.infolist():
        path = (INPUT_ROOT / member.filename).resolve()
        if not path.is_relative_to(INPUT_ROOT.resolve()):
            raise ValueError("Unsafe ZIP member path")
    archive.extractall(INPUT_ROOT)
plan_files = sorted(INPUT_ROOT.glob("*/adaptation_plans/*_adaptation_plan.json"))
assert plan_files, "No Adaptation Plan JSONs detected"
print("Plans found:", len(plan_files))

## 3 · Run safe auto drafts for ALL six Tables

Every Plan is checked independently. A blocked or rebuild-only Table generates a skip report, not a Word output. An eligible text edit must have a verifiable source SHA and bookmarked text; treatment labels additionally require a matching *verified* SAP body quotation.

**No approvals are granted or simulated.** Any generated DOCX is strictly `AUTO_TEXT_PREVIEW_UNAPPROVED`; it is not a final TFL shell.

In [ ]:
OUTPUT_DIR = Path("/content/step4_auto_text_draft_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
records = []
for path in plan_files:
    plan = json.loads(path.read_text(encoding="utf-8"))
    tid = str(plan["target_tfl_id"])
    source_name = str(plan["source_template_id"])
    source_docx = PROJECT_ROOT / "library" / "TFL_Template_Library" / "table" / f"{source_name}.docx"
    dest_folder = OUTPUT_DIR / tid
    dest_folder.mkdir(parents=True, exist_ok=True)
    dest_docx = dest_folder / f"Table_{tid}_AUTO_TEXT_PREVIEW_UNAPPROVED.docx"
    try:
        report = build_unapproved_auto_text_draft(
            plan,
            source_docx=source_docx,
            output_docx=dest_docx,
        )
        report_file = dest_folder / f"{tid}_auto_text_report.json"
        report_file.write_text(
            json.dumps(report, indent=2, ensure_ascii=False)+"\n", encoding="utf-8"
        )
        records.append({
            "TFL": tid, "Gate": report["planning_gate"], "Status": report["status"],
            "Auto preview text edits": report["applied_count"],
            "Unapproved DOCX generated": bool(report["output_docx"]),
            "Remaining operations": sum(report["remaining_operations_by_action"].values()),
            "Target spec gaps": ", ".join(report["target_unresolved_fields"]),
            "Clinical approval": report["clinical_approval"],
        })
        print(f"{tid}: {report['status']} — {report['applied_count']} preview text edits")
        if report["output_docx"]:
            display(FileLink(report["output_docx"], result_html_prefix="UNAPPROVED Word Preview: "))
    except Exception as exc:
        traceback.print_exc()
        records.append({"TFL": tid, "Status": "error",
                        "Error": f"{type(exc).__name__}: {exc}"})

summary = pd.DataFrame(records)
display(summary)
summary.to_csv(OUTPUT_DIR / "auto_text_draft_summary.csv", index=False)
print("Important: Preview success != complete clinical TFL shell generation.")

## 4 · Download all preview drafts and exception reports

In [ ]:
from google.colab import files
archive_path = Path(shutil.make_archive(
    "/content/step4_auto_text_draft_outputs", "zip", root_dir=OUTPUT_DIR
))
print("Archive:", archive_path, "MB:", round(archive_path.stat().st_size/2**20, 2))
files.download(str(archive_path))

## What is and is not automated?

This workflow **automatically proposes and previews** only safe textual differences in a source Word copy. It checks source file integrity, bookmarks, and the presence of target treatment names in verified SAP quotations. It never declares clinical equivalence and never approves edits.

It does **not** automatically insert missing Overall columns, construct target AE rows, adapt statistics, footnotes, or render/inspect the document. These remain unresolved exceptions. A complete end-to-end shell-generation agent will need a separate structural executor and statistical-evidence validation stage.

Notebook 07 remains available for projects that require formal human approval of text edits, but is **not required** for this automatic preview experiment.